In [1]:
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
from transformers.data.data_collator import DataCollatorMixin

from ICL.datasets.gen import UnifiedRHMDataset


@dataclass
class RHMTrainingConfig:
    """Configuration for RHM training with HuggingFace integration."""
    
    # Basic training parameters
    output_dir: str = "./rhm_training_output"
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    
    # Sequence packing parameters
    max_sequence_length: int = 512
    pack_sequences: bool = True
    separator_token_id: int = 0  # Use pad token as separator
    
    # Vocabulary parameters
    vocab_size: int = 32
    pad_token_id: int = 0
    mask_token_id: int = 33  # vocab_size + 1
    
    # Task-specific parameters
    mlm: bool = True  # True for MLM, False for CLM
    mlm_probability: float = 0.15
    
    # Evaluation parameters
    eval_steps: int = 500
    save_steps: int = 1000
    logging_steps: int = 100
    
    # Other parameters
    seed: int = 42
    dataloader_num_workers: int = 4
    remove_unused_columns: bool = False
    
    def to_training_arguments(self) -> TrainingArguments:
        """Convert to HuggingFace TrainingArguments."""
        return TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            per_device_eval_batch_size=self.per_device_eval_batch_size,
            learning_rate=self.learning_rate,
            weight_decay=self.weight_decay,
            warmup_steps=self.warmup_steps,
            eval_steps=self.eval_steps,
            save_steps=self.save_steps,
            logging_steps=self.logging_steps,
            evaluation_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            seed=self.seed,
            dataloader_num_workers=self.dataloader_num_workers,
            remove_unused_columns=self.remove_unused_columns,
            report_to=None,  # Disable wandb by default
        )


def prepare_packed_dataset(
    dataset_path: str,
    config: RHMTrainingConfig,
    train_split_ratio: float = 0.8,
    filter_config_L: int | None = None,
    filter_config_m: int | None = None,
    max_samples: int | None = None
) -> tuple[Dataset, Dataset, dict[str, Any]]:
    """Prepare packed dataset for HuggingFace training.
    
    Args:
        dataset_path: Path to unified RHM dataset
        config: Training configuration
        train_split_ratio: Ratio for train/eval split
        filter_config_L: Filter by hierarchy depth
        filter_config_m: Filter by multiplicity
        max_samples: Maximum samples to use
        
    Returns:
        Tuple of (train_dataset, eval_dataset, metadata)
    """
    print("=" * 60)
    print("PREPARING PACKED DATASET")
    print("=" * 60)
    
    # Load unified dataset
    unified_dataset = UnifiedRHMDataset(dataset_path)
    dataset = unified_dataset.get_dataset()
    
    print(f"Original dataset size: {len(dataset):,}")
    
    # Apply filters if specified
    if filter_config_L is not None or filter_config_m is not None:
        dataset = unified_dataset.filter_by_config(L=filter_config_L, m=filter_config_m)
        print(f"After config filtering: {len(dataset):,}")
    
    # Limit samples if specified
    if max_samples is not None and len(dataset) > max_samples:
        indices = list(range(len(dataset)))
        random.shuffle(indices)
        dataset = dataset.select(indices[:max_samples])
        print(f"After sampling: {len(dataset):,}")
    
    # Prepare sequences for packing
    if config.pack_sequences:
        packed_dataset = _pack_sequences(dataset, config)
    else:
        packed_dataset = _prepare_individual_sequences(dataset, config)
    
    # Split into train/eval
    split_idx = int(len(packed_dataset) * train_split_ratio)
    train_dataset = packed_dataset.select(range(split_idx))
    eval_dataset = packed_dataset.select(range(split_idx, len(packed_dataset)))
    
    metadata = {
        "original_size": len(unified_dataset.get_dataset()),
        "filtered_size": len(dataset),
        "packed_size": len(packed_dataset),
        "train_size": len(train_dataset),
        "eval_size": len(eval_dataset),
        "config": config,
        "vocab_info": unified_dataset.get_vocab_info(),
    }
    
    print(f"Train dataset: {len(train_dataset):,}")
    print(f"Eval dataset: {len(eval_dataset):,}")
    print("=" * 60)
    
    return train_dataset, eval_dataset, metadata


def _pack_sequences(dataset: Dataset, config: RHMTrainingConfig) -> Dataset:
    """Pack multiple sequences into longer training examples with separator tokens.
    
    SEPARATOR INSERTION HAPPENS HERE:
    - Takes individual RHM sequences: [1, 5, 3, 2], [4, 1, 6], [2, 3, 5, 1]
    - Packs with separators: [1, 5, 3, 2, 0, 4, 1, 6, 0, 2, 3, 5, 1]
    - Where 0 is the separator_token_id
    """
    print("Packing sequences with separator tokens...")
    
    packed_examples = []
    current_sequence = []
    current_length = 0
    sequences_in_pack = 0
    
    # Reserve space for separator tokens
    effective_max_length = config.max_sequence_length - 10
    
    for example in dataset:
        sequence = example["input_ids"]
        
        # Skip empty sequences
        if not sequence:
            continue
            
        # If adding this sequence would exceed max length, finalize current packed sequence
        if current_length + len(sequence) + 1 > effective_max_length and current_sequence:
            packed_examples.append({
                "input_ids": current_sequence,
                "length": len(current_sequence),
                "num_sequences_packed": sequences_in_pack,
            })
            current_sequence = []
            current_length = 0
            sequences_in_pack = 0
        
        # *** SEPARATOR INSERTION HAPPENS HERE ***
        # Add separator if this isn't the first sequence in the pack
        if current_sequence:
            current_sequence.append(config.separator_token_id)
            current_length += 1
            print(f"  Added separator token {config.separator_token_id} between sequences")
        
        # Add the actual sequence
        current_sequence.extend(sequence)
        current_length += len(sequence)
        sequences_in_pack += 1
        
        print(f"  Packed sequence {sequences_in_pack}: length {len(sequence)}, total length now {current_length}")
    
    # Add the last packed sequence if it exists
    if current_sequence:
        packed_examples.append({
            "input_ids": current_sequence,
            "length": len(current_sequence),
            "num_sequences_packed": sequences_in_pack,
        })
    
    total_sequences = sum(ex["num_sequences_packed"] for ex in packed_examples)
    print(f"Packed {total_sequences} individual sequences into {len(packed_examples)} training examples")
    avg_length = sum(ex["length"] for ex in packed_examples) / len(packed_examples)
    avg_sequences_per_pack = total_sequences / len(packed_examples)
    print(f"Average packed sequence length: {avg_length:.1f}")
    print(f"Average sequences per pack: {avg_sequences_per_pack:.1f}")
    
    return Dataset.from_list(packed_examples)


def _prepare_individual_sequences(dataset: Dataset, config: RHMTrainingConfig) -> Dataset:
    """Prepare individual sequences without packing."""
    print("Preparing individual sequences...")
    
    examples = []
    for example in dataset:
        sequence = example["input_ids"]
        
        # Skip empty sequences
        if not sequence:
            continue
            
        # Truncate if too long
        if len(sequence) > config.max_sequence_length:
            sequence = sequence[:config.max_sequence_length]
        
        examples.append({
            "input_ids": sequence,
            "length": len(sequence),
        })
    
    print(f"Prepared {len(examples)} individual sequences")
    return Dataset.from_list(examples)


class RHMDataCollator(DataCollatorForLanguageModeling):
    """Extended data collator for RHM with separator token handling."""
    
    def __init__(
        self,
        tokenizer,
        mlm: bool = True,
        mlm_probability: float = 0.15,
        separator_token_id: int = 0,
        return_tensors: str = "pt",
    ):
        """Initialize RHM data collator.
        
        Args:
            tokenizer: Tokenizer (can be None for simple integer sequences)
            mlm: Whether to use masked language modeling
            mlm_probability: Probability of masking tokens
            separator_token_id: ID of separator token
            return_tensors: Format of returned tensors
        """
        # Initialize with dummy tokenizer if None provided
        if tokenizer is None:
            from transformers import PreTrainedTokenizer
            tokenizer = PreTrainedTokenizer()
            tokenizer.pad_token_id = 0
        
        super().__init__(
            tokenizer=tokenizer,
            mlm=mlm,
            mlm_probability=mlm_probability,
            return_tensors=return_tensors,
        )
        self.separator_token_id = separator_token_id
    
    def torch_call(self, examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        """Process batch with separator token awareness."""
        # Convert to format expected by parent class
        batch = []
        for example in examples:
            if isinstance(example["input_ids"], list):
                batch.append({"input_ids": torch.tensor(example["input_ids"])})
            else:
                batch.append({"input_ids": example["input_ids"]})
        
        # Use parent's processing but handle separators specially
        result = super().torch_call(batch)
        
        # Ensure separator tokens are never masked in MLM
        if self.mlm and "labels" in result:
            separator_mask = (result["input_ids"] == self.separator_token_id)
            result["labels"][separator_mask] = -100  # Don't compute loss on separators
        
        return result


class RHMTrainer(Trainer):
    """Simplified RHM trainer using standard HuggingFace components."""
    
    def __init__(
        self,
        model,
        args: TrainingArguments,
        train_dataset: Dataset,
        eval_dataset: Dataset | None = None,
        data_collator: RHMDataCollator | None = None,
        config: RHMTrainingConfig | None = None,
        **kwargs
    ):
        """Initialize RHM trainer.
        
        Args:
            model: The model to train
            args: Training arguments
            train_dataset: Training dataset
            eval_dataset: Evaluation dataset
            data_collator: Data collator
            config: RHM training configuration
            **kwargs: Additional arguments for Trainer
        """
        self.rhm_config = config
        
        # Create default data collator if none provided
        if data_collator is None:
            data_collator = RHMDataCollator(
                tokenizer=None,  # We handle raw integer sequences
                mlm=config.mlm if config else True,
                mlm_probability=config.mlm_probability if config else 0.15,
                separator_token_id=config.separator_token_id if config else 0,
            )
        
        super().__init__(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            **kwargs
        )
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """Compute loss with special handling for RHM sequences."""
        # Standard loss computation - the data collator handles separator masking
        return super().compute_loss(model, inputs, return_outputs)


class HierarchicalMetricsCallback(TrainerCallback):
    """Callback for computing hierarchical-specific metrics during training."""
    
    def __init__(self, eval_dataset: Dataset, config: RHMTrainingConfig):
        """Initialize metrics callback.
        
        Args:
            eval_dataset: Evaluation dataset for computing metrics
            config: Training configuration
        """
        self.eval_dataset = eval_dataset
        self.config = config
        self.step_count = 0
    
    def on_evaluate(self, args, state, control, model, logs=None, **kwargs):
        """Compute additional metrics during evaluation."""
        if logs is None:
            return
        
        # Add custom metrics here
        # For example: perplexity per hierarchy level, separator token accuracy, etc.
        
        # Simple example: track evaluation frequency
        self.step_count += 1
        logs["hierarchical_eval_count"] = self.step_count
        
        print(f"Hierarchical evaluation #{self.step_count} completed")
        print(f"Current eval loss: {logs.get('eval_loss', 'N/A'):.4f}")


def create_rhm_training_pipeline(
    dataset_path: str,
    model,
    config: RHMTrainingConfig,
    **dataset_kwargs
) -> tuple[RHMTrainer, dict[str, Any]]:
    """Create complete RHM training pipeline.
    
    Args:
        dataset_path: Path to unified RHM dataset
        model: Model to train
        config: Training configuration
        **dataset_kwargs: Additional arguments for dataset preparation
        
    Returns:
        Tuple of (trainer, metadata)
    """
    print("Creating RHM training pipeline...")
    
    # Prepare datasets
    train_dataset, eval_dataset, metadata = prepare_packed_dataset(
        dataset_path=dataset_path,
        config=config,
        **dataset_kwargs
    )
    
    # Create training arguments
    training_args = config.to_training_arguments()
    
    # Create data collator
    data_collator = RHMDataCollator(
        tokenizer=None,
        mlm=config.mlm,
        mlm_probability=config.mlm_probability,
        separator_token_id=config.separator_token_id,
    )
    
    # Create callbacks
    callbacks = [
        HierarchicalMetricsCallback(eval_dataset, config)
    ]
    
    # Create trainer
    trainer = RHMTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        config=config,
        callbacks=callbacks,
    )
    
    print("Training pipeline created successfully!")
    return trainer, metadata


# Test function to demonstrate usage
def test_rhm_training_pipeline():
    """Test the RHM training pipeline with a simple model."""
    import torch.nn as nn
    
    # Create simple test model
    class SimpleTransformer(nn.Module):
        def __init__(self, vocab_size: int, d_model: int = 128):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size + 4, d_model)  # +4 for special tokens
            self.transformer = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d_model, nhead=4, batch_first=True),
                num_layers=2
            )
            self.lm_head = nn.Linear(d_model, vocab_size + 4)
        
        def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
            x = self.embedding(input_ids)
            x = self.transformer(x)
            logits = self.lm_head(x)
            
            loss = None
            if labels is not None:
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            
            return {"loss": loss, "logits": logits}
    
    # Create configuration
    config = RHMTrainingConfig(
        output_dir="./test_rhm_output",
        num_train_epochs=1.0,
        per_device_train_batch_size=4,
        max_sequence_length=256,
        pack_sequences=True,
        learning_rate=1e-4,
        eval_steps=50,
        save_steps=100,
    )
    
    # Create model
    model = SimpleTransformer(vocab_size=config.vocab_size)
    
    # Note: This would require actual dataset path
    # trainer, metadata = create_rhm_training_pipeline(
    #     dataset_path="path/to/your/rhm/dataset",
    #     model=model,
    #     config=config,
    #     max_samples=1000  # Use subset for testing
    # )
    
    # trainer.train()
    
    print("Test pipeline created successfully!")
    return config, model


if __name__ == "__main__":
    # Run test
    test_rhm_training_pipeline()

Test pipeline created successfully!


/Users/jliu/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [ ]:
"""Modified RHM Training Configuration with DataLoader Compatibility"""

import json
import logging
import time
import typing as t
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

# Import our custom data collator
from .rhm_dataloader import RHMDataCollator

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


@dataclass
class RHMTrainingConfig:
    """Configuration for RHM training with DataLoader compatibility."""

    # Model configuration
    model_name_or_path: str | None = None
    vocab_size: int = 32  # Base vocabulary size (matches dataloader)
    hidden_size: int = 512
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    intermediate_size: int = 2048
    max_position_embeddings: int = 2048

    # Training configuration
    task_name: str = "clm"  # "clm" or "mlm"
    output_dir: str = "./rhm_training_output"
    num_train_epochs: int = 10
    per_device_train_batch_size: int = 16
    per_device_eval_batch_size: int = 32
    gradient_accumulation_steps: int = 1
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    lr_scheduler_type: str = "linear"

    # Sequence packing configuration (must match dataloader)
    max_sequence_length: int = 512
    pack_sequences: bool = True
    separator_token_id: int = 0  # Pad token used as separator

    # MLM configuration (when task_name="mlm")
    mlm_probability: float = 0.15
    mask_strategy: str = "random"  # "random", "hierarchical", "level_specific"

    # Special token configuration
    pad_token_id: int = 0
    mask_token_id: int = 33  # vocab_size + 1
    cls_token_id: int = 34   # vocab_size + 2
    sep_token_id: int = 35   # vocab_size + 3

    # Checkpoint configuration
    save_strategy: str = "steps"
    save_steps: int = 500
    save_total_limit: int = 5
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "eval_loss"
    greater_is_better: bool = False

    # Evaluation configuration
    evaluation_strategy: str = "steps"
    eval_steps: int = 500
    eval_accumulation_steps: int | None = None

    # Logging configuration
    logging_strategy: str = "steps"
    logging_steps: int = 100
    report_to: list[str] = field(default_factory=lambda: ["tensorboard"])
    run_name: str | None = None

    # Optimization configuration
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0

    # Early stopping
    early_stopping: bool = True
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.0

    # Mixed precision
    fp16: bool = False
    bf16: bool = False

    # Data configuration
    dataloader_num_workers: int = 0
    dataloader_pin_memory: bool = True
    remove_unused_columns: bool = False  # Keep metadata for hierarchical analysis

    # Hierarchical analysis
    track_hierarchical_metrics: bool = True
    hierarchical_eval_frequency: int = 1000

    # Reproducibility
    seed: int = 42

    @property
    def effective_vocab_size(self) -> int:
        """Total vocabulary size including special tokens."""
        return self.vocab_size + 4  # +4 for pad, mask, cls, sep

    def to_training_arguments(self) -> TrainingArguments:
        """Convert to HuggingFace TrainingArguments with version compatibility."""
        training_args_dict = {
            "output_dir": self.output_dir,
            "num_train_epochs": self.num_train_epochs,
            "per_device_train_batch_size": self.per_device_train_batch_size,
            "per_device_eval_batch_size": self.per_device_eval_batch_size,
            "gradient_accumulation_steps": self.gradient_accumulation_steps,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "warmup_ratio": self.warmup_ratio,
            "lr_scheduler_type": self.lr_scheduler_type,
            "save_strategy": self.save_strategy,
            "save_steps": self.save_steps,
            "save_total_limit": self.save_total_limit,
            "load_best_model_at_end": self.load_best_model_at_end,
            "metric_for_best_model": self.metric_for_best_model,
            "greater_is_better": self.greater_is_better,
            "evaluation_strategy": self.evaluation_strategy,
            "eval_steps": self.eval_steps,
            "eval_accumulation_steps": self.eval_accumulation_steps,
            "logging_strategy": self.logging_strategy,
            "logging_steps": self.logging_steps,
            "report_to": self.report_to,
            "run_name": self.run_name,
            "adam_beta1": self.adam_beta1,
            "adam_beta2": self.adam_beta2,
            "adam_epsilon": self.adam_epsilon,
            "max_grad_norm": self.max_grad_norm,
            "fp16": self.fp16,
            "bf16": self.bf16,
            "dataloader_num_workers": self.dataloader_num_workers,
            "dataloader_pin_memory": self.dataloader_pin_memory,
            "remove_unused_columns": self.remove_unused_columns,
            "seed": self.seed,
        }

        # Handle version compatibility for accelerator_config
        try:
            TrainingArguments(output_dir="test", accelerator_config=None)
            training_args_dict["accelerator_config"] = None
            logger.info("Using accelerator_config=None for compatibility")
        except TypeError:
            logger.info("accelerator_config not supported in this transformers version - skipping")

        return TrainingArguments(**training_args_dict)


class HierarchicalMetricsCallback(TrainerCallback):
    """Callback to track hierarchical-specific metrics with packed sequence support."""

    def __init__(self, config: RHMTrainingConfig, dataset_metadata: dict[str, t.Any]):
        self.config = config
        self.dataset_metadata = dataset_metadata
        self.hierarchical_metrics: list[dict[str, t.Any]] = []

    def on_evaluate(self, args, state, control, model, eval_dataloader, **kwargs):
        """Called after evaluation."""
        if not self.config.track_hierarchical_metrics:
            return

        if state.global_step % self.config.hierarchical_eval_frequency == 0:
            metrics = self._compute_hierarchical_metrics(model, eval_dataloader)
            self.hierarchical_metrics.append({
                "step": state.global_step, 
                "epoch": state.epoch, 
                "metrics": metrics
            })

            # Log metrics
            logger.info(f"Hierarchical metrics at step {state.global_step}: {metrics}")

    def _compute_hierarchical_metrics(self, model, dataloader) -> dict[str, float]:
        """Compute metrics specific to hierarchical configurations with packed sequences."""
        model.eval()

        total_loss = 0.0
        total_samples = 0
        separator_token_count = 0
        separator_accuracy = 0.0

        with torch.no_grad():
            for batch in dataloader:
                # Move batch to device
                device = next(model.parameters()).device

                # Prepare batch for model
                model_inputs = {}
                for key in ["input_ids", "attention_mask", "labels"]:
                    if key in batch and batch[key] is not None:
                        if hasattr(batch[key], "to"):
                            model_inputs[key] = batch[key].to(device)
                        else:
                            model_inputs[key] = batch[key]

                outputs = model(**model_inputs)
                
                if outputs.loss is not None:
                    total_loss += outputs.loss.item()
                    total_samples += 1

                # Analyze separator tokens if present
                if "input_ids" in model_inputs:
                    input_ids = model_inputs["input_ids"]
                    separator_mask = (input_ids == self.config.separator_token_id)
                    separator_token_count += separator_mask.sum().item()

                    # If we have predictions, check separator prediction accuracy
                    if hasattr(outputs, "logits") and outputs.logits is not None:
                        predictions = torch.argmax(outputs.logits, dim=-1)
                        separator_predictions = predictions[separator_mask]
                        correct_separators = (separator_predictions == self.config.separator_token_id).sum().item()
                        if separator_mask.sum().item() > 0:
                            separator_accuracy += correct_separators / separator_mask.sum().item()

        # Compute hierarchical metrics
        hierarchical_metrics = {}
        
        if total_samples > 0:
            hierarchical_metrics["avg_loss"] = total_loss / total_samples
            hierarchical_metrics["total_sequences_evaluated"] = total_samples
        
        if separator_token_count > 0:
            hierarchical_metrics["separator_tokens_per_sequence"] = separator_token_count / total_samples
            hierarchical_metrics["separator_prediction_accuracy"] = separator_accuracy / total_samples

        # Add packing-related metrics
        if self.config.pack_sequences:
            hierarchical_metrics["packing_enabled"] = True
            hierarchical_metrics["avg_packed_length"] = self.dataset_metadata.get("avg_seq_length", 0)
        
        model.train()
        return hierarchical_metrics


class CheckpointCallback(TrainerCallback):
    """Enhanced checkpoint callback with hierarchical metadata and packing info."""

    def __init__(self, config: RHMTrainingConfig, dataset_metadata: dict[str, t.Any]):
        self.config = config
        self.dataset_metadata = dataset_metadata
        self.checkpoint_history: list[dict[str, t.Any]] = []

    def on_save(self, args, state, control, model, tokenizer, **kwargs):
        """Called when saving checkpoint."""
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"

        # Save enhanced metadata with packing information
        enhanced_metadata = {
            "training_config": self.config.__dict__,
            "dataset_metadata": self.dataset_metadata,
            "packing_info": {
                "pack_sequences": self.config.pack_sequences,
                "separator_token_id": self.config.separator_token_id,
                "max_sequence_length": self.config.max_sequence_length,
                "effective_vocab_size": self.config.effective_vocab_size,
            },
            "training_state": {
                "global_step": state.global_step,
                "epoch": state.epoch,
                "learning_rate": state.log_history[-1].get("learning_rate", 0) if state.log_history else 0,
                "train_loss": state.log_history[-1].get("train_loss", 0) if state.log_history else 0,
                "eval_loss": state.log_history[-1].get("eval_loss", 0) if state.log_history else 0,
            },
            "model_config": model.config.to_dict() if hasattr(model.config, "to_dict") else str(model.config),
            "timestamp": datetime.now().isoformat(),
        }

        # Save metadata
        metadata_path = checkpoint_dir / "training_metadata.json"
        metadata_path.parent.mkdir(parents=True, exist_ok=True)
        with metadata_path.open("w") as f:
            json.dump(enhanced_metadata, f, indent=2, default=str)

        # Track checkpoint
        self.checkpoint_history.append({
            "step": state.global_step,
            "epoch": state.epoch,
            "path": str(checkpoint_dir),
            "timestamp": datetime.now().isoformat(),
        })

        logger.info(f"Enhanced checkpoint saved to {checkpoint_dir}")


class RHMTrainer:
    """Main trainer class for RHM models with DataLoader compatibility."""

    def __init__(
        self,
        training_config: RHMTrainingConfig,
        train_dataset: Dataset,
        eval_dataset: Dataset,
        dataset_metadata: dict[str, t.Any],
        data_collator: RHMDataCollator | None = None,
    ):
        """Initialize RHM trainer with Dataset objects.

        Args:
            training_config: Training configuration
            train_dataset: HuggingFace Dataset for training
            eval_dataset: HuggingFace Dataset for evaluation
            dataset_metadata: Metadata from dataset preparation
            data_collator: Custom data collator (will create default if None)

        """
        self.config = training_config
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.dataset_metadata = dataset_metadata

        # Validate configuration compatibility
        self._validate_config_compatibility()

        # Set up output directory
        self.output_dir = Path(self.config.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Initialize model
        self.model = self._create_model()

        # Set up data collator
        self.data_collator = data_collator or self._create_data_collator()

        # Set up training arguments
        self.training_args = self.config.to_training_arguments()

        # Set up callbacks
        self.callbacks = self._setup_callbacks()

        # Initialize trainer placeholder
        self.trainer: Trainer | None = None

        logger.info(f"RHM Trainer initialized for task: {self.config.task_name}")
        logger.info(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        logger.info(f"Effective vocabulary size: {self.config.effective_vocab_size}")
        logger.info(f"Sequence packing: {self.config.pack_sequences}")
        logger.info(f"Separator token ID: {self.config.separator_token_id}")
        device_name = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"
        logger.info(f"Training on device: {device_name}")

    def _validate_config_compatibility(self) -> None:
        """Validate that training config is compatible with dataset metadata."""
        if "separator_token_id" in self.dataset_metadata:
            dataset_sep_id = self.dataset_metadata["separator_token_id"]
            if dataset_sep_id != self.config.separator_token_id:
                logger.warning(
                    f"Separator token ID mismatch: config={self.config.separator_token_id}, "
                    f"dataset={dataset_sep_id}. Using config value."
                )

        if "packing_enabled" in self.dataset_metadata:
            dataset_packing = self.dataset_metadata["packing_enabled"]
            if dataset_packing != self.config.pack_sequences:
                logger.warning(
                    f"Packing mismatch: config={self.config.pack_sequences}, "
                    f"dataset={dataset_packing}"
                )

    def _create_model(self):
        """Create model based on task and configuration with proper vocabulary size."""
        # Create model configuration with correct vocabulary size
        if self.config.model_name_or_path:
            # Load from existing model
            model_config = AutoConfig.from_pretrained(self.config.model_name_or_path)
            model_config.vocab_size = self.config.effective_vocab_size
        elif self.config.task_name == "clm":
            from transformers import GPT2Config

            model_config = GPT2Config(
                vocab_size=self.config.effective_vocab_size,
                n_positions=self.config.max_position_embeddings,
                n_embd=self.config.hidden_size,
                n_layer=self.config.num_hidden_layers,
                n_head=self.config.num_attention_heads,
                n_inner=self.config.intermediate_size,
                resid_pdrop=0.1,
                embd_pdrop=0.1,
                attn_pdrop=0.1,
                use_cache=False,  # Disable for training
                pad_token_id=self.config.pad_token_id,
            )
        elif self.config.task_name == "mlm":
            from transformers import BertConfig

            model_config = BertConfig(
                vocab_size=self.config.effective_vocab_size,
                hidden_size=self.config.hidden_size,
                num_hidden_layers=self.config.num_hidden_layers,
                num_attention_heads=self.config.num_attention_heads,
                intermediate_size=self.config.intermediate_size,
                max_position_embeddings=self.config.max_position_embeddings,
                hidden_dropout_prob=0.1,
                attention_probs_dropout_prob=0.1,
                pad_token_id=self.config.pad_token_id,
                mask_token_id=self.config.mask_token_id,
                cls_token_id=self.config.cls_token_id,
                sep_token_id=self.config.sep_token_id,
            )
        else:
            raise ValueError(f"Unknown task: {self.config.task_name}")

        # Create model
        if self.config.model_name_or_path:
            if self.config.task_name == "clm":
                model = AutoModelForCausalLM.from_pretrained(
                    self.config.model_name_or_path, config=model_config
                )
            else:
                model = AutoModelForMaskedLM.from_pretrained(
                    self.config.model_name_or_path, config=model_config
                )
        elif self.config.task_name == "clm":
            model = AutoModelForCausalLM.from_config(model_config)
        else:
            model = AutoModelForMaskedLM.from_config(model_config)

        # Move to device
        if torch.cuda.is_available():
            model = model.cuda()

        return model

    def _create_data_collator(self) -> RHMDataCollator:
        """Create data collator compatible with our dataset format."""
        return RHMDataCollator(
            tokenizer=None,  # We work with raw integer sequences
            mlm=(self.config.task_name == "mlm"),
            mlm_probability=self.config.mlm_probability,
            separator_token_id=self.config.separator_token_id,
            return_tensors="pt",
        )

    def _setup_callbacks(self) -> list[TrainerCallback]:
        """Set up training callbacks."""
        callbacks = []

        # Hierarchical metrics callback
        if self.config.track_hierarchical_metrics:
            callbacks.append(HierarchicalMetricsCallback(self.config, self.dataset_metadata))

        # Enhanced checkpoint callback
        callbacks.append(CheckpointCallback(self.config, self.dataset_metadata))

        # Early stopping callback
        if self.config.early_stopping:
            callbacks.append(
                EarlyStoppingCallback(
                    early_stopping_patience=self.config.early_stopping_patience,
                    early_stopping_threshold=self.config.early_stopping_threshold,
                )
            )

        return callbacks

    def train(self) -> dict[str, t.Any]:
        """Run training with improved error handling."""
        logger.info("Starting training...")
        logger.info(f"Task: {self.config.task_name}")
        logger.info(f"Training samples: {len(self.train_dataset):,}")
        logger.info(f"Evaluation samples: {len(self.eval_dataset):,}")
        logger.info(f"Epochs: {self.config.num_train_epochs}")
        logger.info(f"Batch size: {self.config.per_device_train_batch_size}")
        logger.info(f"Learning rate: {self.config.learning_rate}")
        logger.info(f"Packed sequences: {self.config.pack_sequences}")

        # Save initial configuration
        self._save_training_setup()

        # Initialize trainer with datasets and data collator
        self.trainer = Trainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            data_collator=self.data_collator,
            callbacks=self.callbacks,
        )

        # Start training
        start_time = time.time()

        try:
            train_result = self.trainer.train()
            training_time = time.time() - start_time

            # Save final model
            self.trainer.save_model()

            # Collect final metrics
            final_metrics = {
                "train_result": train_result.metrics,
                "training_time": training_time,
                "final_eval_metrics": self.trainer.evaluate(),
                "model_size": sum(p.numel() for p in self.model.parameters()),
                "dataset_info": self.dataset_metadata,
                "effective_vocab_size": self.config.effective_vocab_size,
                "packing_info": {
                    "pack_sequences": self.config.pack_sequences,
                    "separator_token_id": self.config.separator_token_id,
                    "max_sequence_length": self.config.max_sequence_length,
                },
            }

            # Save final metrics
            metrics_path = self.output_dir / "final_metrics.json"
            with metrics_path.open("w") as f:
                json.dump(final_metrics, f, indent=2, default=str)

            logger.info(f"Training completed in {training_time:.2f} seconds")
            logger.info(f"Final evaluation loss: {final_metrics['final_eval_metrics']['eval_loss']:.4f}")

            return final_metrics

        except Exception as e:
            logger.error(f"Training failed: {e}")
            # Save error information
            error_info = {
                "error": str(e),
                "error_type": type(e).__name__,
                "timestamp": datetime.now().isoformat(),
                "training_config": self.config.__dict__,
                "dataset_metadata": self.dataset_metadata,
            }

            error_path = self.output_dir / "training_error.json"
            with error_path.open("w") as f:
                json.dump(error_info, f, indent=2, default=str)

            raise

    def _save_training_setup(self) -> None:
        """Save complete training setup for reproducibility."""
        setup_info = {
            "training_config": self.config.__dict__,
            "dataset_metadata": self.dataset_metadata,
            "model_config": (
                self.model.config.to_dict() if hasattr(self.model.config, "to_dict") else str(self.model.config)
            ),
            "training_arguments": self.training_args.to_dict(),
            "data_collator_info": {
                "type": type(self.data_collator).__name__,
                "mlm": getattr(self.data_collator, "mlm", None),
                "mlm_probability": getattr(self.data_collator, "mlm_probability", None),
                "separator_token_id": getattr(self.data_collator, "separator_token_id", None),
            },
            "device_info": {
                "cuda_available": torch.cuda.is_available(),
                "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
                "cuda_device_name": (torch.cuda.get_device_name() if torch.cuda.is_available() else None),
            },
            "pytorch_version": torch.__version__,
            "timestamp": datetime.now().isoformat(),
        }

        setup_path = self.output_dir / "training_setup.json"
        with setup_path.open("w") as f:
            json.dump(setup_info, f, indent=2, default=str)

        logger.info(f"Training setup saved to {setup_path}")


# Factory function for easy integration
def create_rhm_trainer_from_dataloader(
    dataset_path: str,
    training_config: RHMTrainingConfig,
    **dataset_kwargs
) -> tuple[RHMTrainer, dict[str, t.Any]]:
    """Create RHM trainer using our dataloader factory.
    
    Args:
        dataset_path: Path to RHM dataset
        training_config: Training configuration
        **dataset_kwargs: Arguments for dataset preparation
        
    Returns:
        Tuple of (trainer, combined_metadata)
    """
    from .rhm_dataloader import prepare_packed_dataset
    
    # Prepare datasets using our dataloader
    train_dataset, eval_dataset, dataset_metadata = prepare_packed_dataset(
        dataset_path=dataset_path,
        config=training_config,  # Use training config for consistency
        **dataset_kwargs
    )
    
    # Create trainer
    trainer = RHMTrainer(
        training_config=training_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_metadata=dataset_metadata,
    )
    
    # Combine metadata
    combined_metadata = {
        "dataset_metadata": dataset_metadata,
        "training_config": training_config.__dict__,
    }
    
    return trainer, combined_metadata


# Test function
def test_integrated_rhm_training():
    """Test the complete integrated training pipeline."""
    # Create configuration
    config = RHMTrainingConfig(
        task_name="clm",
        vocab_size=32,
        output_dir="./test_integrated_rhm",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        max_sequence_length=256,
        pack_sequences=True,
        learning_rate=1e-4,
        eval_steps=50,
        save_steps=100,
    )
    
    # Note: This would require actual dataset path
    # trainer, metadata = create_rhm_trainer_from_dataloader(
    #     dataset_path="path/to/your/rhm/dataset",
    #     training_config=config,
    #     max_samples=1000  # Use subset for testing
    # )
    
    # trainer.train()
    
    print("Test integrated training pipeline created successfully!")
    return config


if __name__ == "__main__":
    # Run test
    test_integrated_rhm_training()